# CPP：CIFAR-10 官方实验复现

目标：使用官方 CPP 模型与损失，在 CIFAR-10 的 50,000 张训练图像上复现论文的聚类实验。先用冻结的 CLIP ViT-L/14 提取 768 维特征并缓存，再训练 CPP；这对应官方仓库提供的“预提取特征”路径。论文 Table 1 报告 CIFAR-10 ACC 97.4%、NMI 93.6%。

官方源码固定在 commit `cb39bdc65f346b4cb5950d14af2e393a5f6a3566`。[论文](https://arxiv.org/html/2306.05272v5)、[官方仓库](https://github.com/LeslieTrue/CPP)。

**实验设置差异**：论文附录 C 对 CIFAR-10 写的是总计 5 轮、初始化 1 轮；官方 README 示例命令写 15 轮、预热 50 步。本 notebook 采用论文附录 C 的轮次。官方训练脚本只在训练批次上即时评分；这里也记录批次分数，因此它不能直接等同于全体 50,000 张图像的一次统一聚类结果。

## 环境

本项目当前的 PyTorch 是 CPU 版 2.5.1，本机没有 CUDA。完整实验需要可用的 GPU 和与之匹配的 PyTorch；请在相应环境中安装官方 OpenAI CLIP、torchvision，以及本 notebook 所用的 scipy、scikit-learn 和 tqdm。不要机械安装官方旧版 `requirements.txt` 覆盖当前项目环境。

本 notebook 使用 `vendor/CPP` 中的官方网络、损失和聚类指标代码；若在新机器运行，先克隆官方仓库并检出上述 commit。CIFAR-10 与 CLIP 权重会在首次运行时下载；特征缓存放在 `data/cpp/`，训练权重放在 `models/`。

?????????????

```bash
git clone https://github.com/LeslieTrue/CPP.git vendor/CPP
git -C vendor/CPP checkout cb39bdc65f346b4cb5950d14af2e393a5f6a3566
```

???? GPU Python ?????? PyTorch ??? `torchvision`???? OpenAI ?? `CLIP`?`pip install git+https://github.com/openai/CLIP.git`??? notebook ??????????


In [ ]:
from pathlib import Path
from subprocess import check_output
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

root = Path.cwd().parent
cpp_dir = root / "vendor" / "CPP"
sys.path.insert(0, str(cpp_dir))

from model.CPP_model import CPPNet
from model.sink_distance import SinkhornDistance
from loss.loss_fn import MLCLoss, TotalCodingRate
from metrics.clustering import spectral_clustering_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python {sys.version.split()[0]}, PyTorch {torch.__version__}, device {device}")
cpp_commit = check_output(["git", "-C", str(cpp_dir), "rev-parse", "HEAD"], text=True).strip()
print(f"CPP source: {cpp_dir}, commit {cpp_commit}")
assert cpp_commit == "cb39bdc65f346b4cb5950d14af2e393a5f6a3566"

## 1. 提取并缓存 CLIP 特征

沿用官方的 CIFAR-10 读取与 CLIP 图像预处理。CLIP 主干不训练，缓存特征后后续运行不再重复编码图像。标签仅在训练期间计算评估指标。

In [ ]:
feature_path = root / "data" / "cpp" / "cifar10_clip_vitl14.pt"
feature_path.parent.mkdir(parents=True, exist_ok=True)

if not feature_path.exists():
    import clip
    from data.dataset import load_dataset

    dataset = load_dataset("cifar10", train=True, path=str(root / "data" / "cpp"))
    image_loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0)
    clip_model, _ = clip.load("ViT-L/14", device=device)
    clip_model.eval()

    features, labels = [], []
    with torch.inference_mode():
        for images, targets in image_loader:
            encoded = clip_model.encode_image(images.to(device))
            features.append(encoded.float().cpu())
            labels.append(targets)
    torch.save({"features": torch.cat(features), "ys": torch.cat(labels)}, feature_path)
    del clip_model

feature_data = torch.load(feature_path, map_location="cpu", weights_only=False)
features = feature_data["features"].float()
labels = feature_data["ys"].long()
print("features:", tuple(features.shape), "labels:", tuple(labels.shape))
assert features.shape == (50_000, 768)
assert labels.shape == (50_000,)

## 2. 训练 CPP

直接使用官方的 `CPPNet`、`SinkhornDistance`、`MLCLoss` 与 `TotalCodingRate`。第一轮只训练特征分支，再将其权重复制给关联分支；其余四轮共同训练。超参数取论文附录 C 的 CIFAR-10 设置和官方训练入口中的损失参数。为方便阅读，这里把官方按“步数”计算的预热写成完整的第一轮。

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

batch_size = 1024
epochs = 5
loader = DataLoader(
    TensorDataset(features, labels),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

model = CPPNet(input_dim=768, hidden_dim=4096, z_dim=128).to(device)
sinkhorn = SinkhornDistance(0.175, max_iter=5)
warmup_loss = TotalCodingRate(eps=0.1)
cpp_loss = MLCLoss(eps=0.1, gamma=1.0)
feature_optimizer = torch.optim.SGD(
    list(model.pre_feature.parameters()) + list(model.subspace.parameters()),
    lr=1e-4, momentum=0.9, weight_decay=1e-4,
)
cluster_optimizer = torch.optim.SGD(
    model.cluster.parameters(), lr=1e-4, momentum=0.9, weight_decay=5e-3,
)

evaluation = []
started = time.perf_counter()

for epoch in range(epochs):
    model.train()
    for step, (x, y) in enumerate(loader, start=1):
        x = x.to(device)
        with torch.autocast(device_type="cuda", enabled=device.type == "cuda"):
            z, codes = model(x)
            similarity = (codes @ codes.T).abs().unsqueeze(0)
            membership = sinkhorn(similarity)[0][0] * len(x)

            if epoch == 0:
                loss = warmup_loss(z)
            else:
                loss, _ = cpp_loss(z, membership)
                loss = loss + 0.05 * 0.5 * membership.norm().square()

        feature_optimizer.zero_grad()
        cluster_optimizer.zero_grad()
        loss.backward()
        feature_optimizer.step()
        if epoch > 0:
            cluster_optimizer.step()

        if epoch > 0 and step % 10 == 0:
            acc, nmi, _, _, _ = spectral_clustering_metrics(
                membership.detach().cpu().numpy(), 10, y.numpy(), verbose=False
            )
            evaluation.append({
                "epoch": epoch + 1, "step": step,
                "acc": float(np.mean(acc)), "nmi": float(np.mean(nmi)),
            })

    if epoch == 0:
        model.cluster.load_state_dict(model.subspace.state_dict())
    print(f"epoch {epoch + 1}/{epochs}, loss {loss.item():.4f}")

elapsed = time.perf_counter() - started
print(f"training seconds: {elapsed:.1f}")

## 3. 汇总与保存

官方指标函数在单个训练批次上做谱聚类，再用真实标签计算准确率和 NMI。下面的均值是**批次评估均值**，用于核对训练是否接近论文的 97.4% / 93.6%；它不是全数据集的一次聚类评分。保留每次评估，避免只挑最高分。

In [ ]:
scores = pd.DataFrame(evaluation)
display(scores.groupby("epoch")[["acc", "nmi"]].mean())

model_path = root / "models" / "cpp_cifar10_paper_settings.pt"
score_path = root / "results" / "cpp_cifar10_batch_scores.csv"
model_path.parent.mkdir(parents=True, exist_ok=True)
score_path.parent.mkdir(parents=True, exist_ok=True)

torch.save({
    "model": model.cpu().state_dict(),
    "cpp_commit": "cb39bdc65f346b4cb5950d14af2e393a5f6a3566",
    "seed": seed,
    "epochs": epochs,
    "batch_size": batch_size,
    "training_seconds": elapsed,
}, model_path)
scores.to_csv(score_path, index=False)
print(model_path, score_path)